# Stroke Model Training and Evaluation

Train the preprocessing and SMOTE/XGBoost workflow, evaluate it on a held-out test set, and inspect feature explanations.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import xgboost as xgb
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
from sklearn.model_selection import train_test_split

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
sys.path.insert(0, str(BASE_DIR))

from src.preprocessing import create_preprocessor

DATA_PATH = BASE_DIR / "data" / "raw" / "full_filled_stroke_data (1).csv"
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
X = df.drop(columns=["stroke"])
y = df["stroke"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = create_preprocessor()
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_proc, y_train)
print(f"Original training distribution: {y_train.value_counts().to_dict()}")
print(f"SMOTE training distribution: {y_train_res.value_counts().to_dict()}")

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    eval_metric="logloss",
    random_state=42
)
model.fit(X_train_res, y_train_res)

predictions = model.predict(X_test_proc)
probabilities = model.predict_proba(X_test_proc)[:, 1]
print(classification_report(y_test, predictions, zero_division=0))
print(f"ROC-AUC: {roc_auc_score(y_test, probabilities):.3f}")

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, predictions),
    display_labels=["No stroke", "Stroke"]
).plot(cmap="Blues")
plt.title("Held-out test confusion matrix")
plt.show()

In [ ]:
import joblib
import shap

MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, MODEL_DIR / "preprocessor.pkl")
joblib.dump(model, MODEL_DIR / "xgboost_stroke_model.pkl")

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_proc)
feature_names = preprocessor.get_feature_names_out()
shap.summary_plot(shap_values, X_test_proc, feature_names=feature_names, show=False)
plt.title("SHAP feature impact on stroke prediction")
plt.tight_layout()
plt.show()
print("Saved preprocessor and XGBoost model artifacts.")